# 02 Procesamiento de texto, recuperación secuencial e indexación

## Objetivo

Implementar y comparar dos mecanismos de recuperación de información —búsqueda secuencial e índice invertido— sobre colecciones de libros del Proyecto Gutenberg, investigando experimentalmente su rendimiento y escalabilidad.

### Parte A. Corpus pequeño

**1. Preparación del corpus**

Utilice los 18 documentos de la colección Gutenberg disponible en NLTK.

1. Cargue los textos originales.
2. Identifique el número de documentos y sus tamaños.
3. Implemente tokenización y normalización: conversión a minúsculas, tratamiento de puntuación y caracteres especiales.

Seleccione y documente una configuración de preprocesamiento que se utilizará consistentemente en todos los experimentos.


**Nombre:** Harry Guajan  
**Fecha:** 13 de octubre de 2026

In [1]:
import re
from collections import Counter
from time import perf_counter_ns

import nltk
import pandas as pd
from IPython.display import display

try:
    from nltk.corpus import gutenberg
    document_ids = gutenberg.fileids()
    if len(document_ids) != 18:
        raise LookupError("La colección Gutenberg de NLTK no está disponible.")
except LookupError:
    nltk.download("gutenberg", quiet=True)
    from nltk.corpus import gutenberg
    document_ids = gutenberg.fileids()

# Que se considera un token valido
token_pattern = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")
# Normalizacion de texto y tokenizacion 
def normalize(text):
    return token_pattern.findall(text.lower())

original_documents = {doc_id: gutenberg.raw(doc_id) for doc_id in document_ids}
tokenized_documents = {
    doc_id: normalize(text) for doc_id, text in original_documents.items()
}

document_summary = pd.DataFrame([
    {
        "document_id": doc_id,
        "original_characters": len(original_documents[doc_id]),
        "tokens": len(tokenized_documents[doc_id]),
    }
    for doc_id in document_ids
])

print(f"Documentos cargados: {len(document_ids)}")
print(f"Tokens totales: {document_summary['tokens'].sum():,}")
display(document_summary)


Documentos cargados: 18
Tokens totales: 2,182,070


,document_id,original_characters,tokens
0,austen-emma.txt,887071,160999
1,austen-persuasion.txt,466292,83660
2,austen-sense.txt,673022,119965
3,bible-kjv.txt,4332554,852256
4,blake-poems.txt,38153,6839
5,bryant-stories.txt,249439,46321
6,burgess-busterbrown.txt,84663,15969
7,carroll-alice.txt,144395,26688
8,chesterton-ball.txt,457450,82252
9,chesterton-brown.txt,406629,72391


**2. Búsqueda secuencial**

Implemente una función que reciba un término de consulta y devuelva los identificadores de todos los documentos que lo contienen con una búsqueda secuencial.

La implementación deberá utilizar obligatoriamente dos lazos `for` anidados: el primero recorre los documentos y el segundo los tokens de cada documento.

No es necesario utilizar índices invertidos, motores de búsqueda ni estructuras auxiliares que almacenen anticipadamente los documentos asociados a cada término.

Ejecute consultas con términos frecuentes, poco frecuentes e inexistentes.

In [14]:
def sequential_search(query):
    #Devuelve los identificadores de los documentos que contienen el término
    query_tokens = normalize(query)

    if len(query_tokens) != 1:
        raise ValueError("La búsqueda secuencial recibe exactamente un término.")

    term = query_tokens[0]
    matches = []

    for document_id, tokens in tokenized_documents.items():
        for token in tokens:
            if token == term:
                matches.append(document_id)
                break

    return matches

# Frecuencia total de apariciones de cada término.
term_counts = Counter(
    token
    for tokens in tokenized_documents.values()
    for token in tokens
)

# Número de documentos distintos en los que aparece cada término.
document_frequency = Counter(
    token
    for tokens in tokenized_documents.values()
    for token in set(tokens)
)

# Consulta frecuente: término presente en más documentos.
#most_common(1): obtiene el término más frecuente por número de documentos.
#[0]: obtiene la primera tupla de resultados.
#[0]: obtiene el término de esa tupla, descartando su frecuencia.

frequent_term = document_frequency.most_common(1)[0][0]

# Consulta poco frecuente: término presente en un solo documento,
# si existe alguno que cumpla los filtros.
rare_candidates = [
    term
    for term, frequency in document_frequency.items()
    if frequency == 1 and term.isalpha() and len(term) > 2
]

if rare_candidates:
    infrequent_term = min(
        rare_candidates,
        key=lambda term: (term_counts[term], term)
    )
else:
    # Si no hay términos presentes en un solo documento,
    # seleccionar el de menor frecuencia documental.
    infrequent_term = min(
        (
            term for term in document_frequency
            if term.isalpha() and len(term) > 2
        ),
        key=lambda term: (
            document_frequency[term],
            term_counts[term],
            term
        )
    )

# Consulta inexistente: un solo token que no aparece en el corpus.
missing_term = "Insaneador"

# Ejecutar las consultas y mostrar sus resultados.
queries = [
    ("frecuente", frequent_term),
    ("poco frecuente", infrequent_term),
    ("inexistente", missing_term)
]

sequential_examples = pd.DataFrame([
    {
        "query": label,
        "term": term,
        "document_frequency": document_frequency.get(term, 0),
        "documents": sequential_search(term)
    }
    for label, term in queries
])

display(sequential_examples)

,query,term,document_frequency,documents
0,frecuente,lost,18,"[austen-emma.txt, austen-persuasion.txt, auste..."
1,poco frecuente,abaddon,1,[bible-kjv.txt]
2,inexistente,Insaneador,0,[]


**3. Medición de tiempos de ejecución**

Utilice `time.perf_counter_ns()` o una herramienta de medición equivalente para obtener el tiempo de ejecución de las consultas.

Cada medición deberá corresponder a **30 ejecuciones de la misma consulta**. Y se debe calcular la media aritmética.

Registre los resultados en milisegundos. No incluya la carga de archivos ni la tokenización dentro del tiempo de consulta.

In [19]:
def mean_query_time(query, search_function, repetitions=30):
    elapsed = []
    for _ in range(repetitions):
        start = perf_counter_ns()
        search_function(query)
        elapsed.append(perf_counter_ns() - start)
    return sum(elapsed) / repetitions / 1_000_000

sequential_timing = pd.DataFrame([
    {
        "query_type": label,
        "term": term,
        "mean_time_ms": mean_query_time(term, sequential_search),
        "repetitions": 30,
    }
    for label, term in [
        ("frecuente", frequent_term),
        ("poco frecuente", infrequent_term),
        ("inexistente", missing_term),
    ]
])
display(sequential_timing)


,query_type,term,mean_time_ms,repetitions
0,frecuente,lost,2.124753,30
1,poco frecuente,abaddon,27.746187,30
2,inexistente,Insaneador,27.084770,30


#### Observacion
Esperaba que los tiempos fueran similares, pero tiene lógica que el tiempo para encontrar el término frecuente sea menor, aunque esto no siempre tiene que ser así. Puede que lost aparezca más veces, pero que se encuentre después de la primera aparición de abaddon. Por lo tanto, esto no comprueba que los términos frecuentes sean necesariamente más rápidos.

Por otro lado, entre el término poco frecuente y el inexistente hay poca diferencia, ya que el algoritmo recorre casi todos los documentos en ambos casos. Además, cuando el término no existe, debe recorrerlos todos.

**4. Construcción del índice invertido**

Implemente un índice invertido que contenga el vocabulario de la colección y las listas de documentos asociados a cada término.

Inspeccione las listas de postings y calcule la frecuencia documental (DF) de varios términos.

Implemente consultas de un término y consultas booleanas con AND, OR y NOT.

In [ ]:
inverted_index = {}
for document_id, tokens in tokenized_documents.items():
    for token in set(tokens):
        inverted_index.setdefault(token, set()).add(document_id)

all_documents = set(document_ids)

def indexed_search(query):
    """Return postings for one normalized term."""
    query_tokens = normalize(query)
    if len(query_tokens) != 1:
        raise ValueError("La consulta recibe exactamente un término.")
    return sorted(inverted_index.get(query_tokens[0], set()))

def boolean_and(left, right):
    return sorted(inverted_index.get(left, set()) & inverted_index.get(right, set()))

def boolean_or(left, right):
    return sorted(inverted_index.get(left, set()) | inverted_index.get(right, set()))

def boolean_not(term):
    return sorted(all_documents - inverted_index.get(term, set()))

postings_terms = [frequent_term, infrequent_term, "the"]
postings = pd.DataFrame([
    {"term": term, "df": len(inverted_index.get(term, set())), "postings": indexed_search(term)}
    for term in dict.fromkeys(postings_terms)
])
display(postings)

boolean_examples = pd.DataFrame([
    {"query": f"{frequent_term} AND {infrequent_term}", "documents": boolean_and(frequent_term, infrequent_term)},
    {"query": f"{frequent_term} OR {infrequent_term}", "documents": boolean_or(frequent_term, infrequent_term)},
    {"query": f"NOT {infrequent_term}", "documents": boolean_not(infrequent_term)},
])
display(boolean_examples)


**5. Comparación**

Ejecute las mismas consultas con ambos métodos.

Compruebe que recuperan exactamente los mismos identificadores de documentos y mida el tiempo medio de 30 ejecuciones para cada consulta y método.

In [ ]:
comparison_rows = []
for label, term in [
    ("frecuente", frequent_term),
    ("poco frecuente", infrequent_term),
    ("inexistente", missing_term),
]:
    sequential_result = sequential_search(term)
    indexed_result = indexed_search(term)
    comparison_rows.extend([
        {
            "query_type": label,
            "term": term,
            "method": "secuencial",
            "documents": sequential_result,
            "mean_time_ms": mean_query_time(term, sequential_search),
        },
        {
            "query_type": label,
            "term": term,
            "method": "índice invertido",
            "documents": indexed_result,
            "mean_time_ms": mean_query_time(term, indexed_search),
        },
    ])
    assert sequential_result == indexed_result

comparison = pd.DataFrame(comparison_rows)
print("Todas las consultas devuelven exactamente los mismos identificadores.")
display(comparison)


### Parte B. Corpus grande

**6. Descarga de Gutenberg**

Descargue el archivo oficial `txt-files.tar.zip`, de aproximadamente 11 GB, desde:

https://www.gutenberg.org/cache/epub/feeds/

Identifique los archivos TXT, evite contar distintas versiones del mismo libro como documentos independientes y registre la cantidad de documentos válidos.

**7. Recuperación sobre el corpus grande**

Reutilice las funciones desarrolladas en la Parte A.

1. Procese los documentos con la misma configuración de normalización.
2. Ejecute las consultas mediante búsqueda secuencial y mida los tiempos.
3. Construya el índice invertido.
4. Repita las consultas utilizando el índice.
5. Compruebe la igualdad de resultados y compare los tiempos.

Trabaje con subconjuntos de la colección de tamaños crecientes, conservando los identificadores originales. Documente los recursos de memoria y almacenamiento utilizados.

### Parte C. Análisis experimental de escalabilidad

**8. Tiempo de ejecución en función del corpus**

Seleccione al menos cinco tamaños crecientes del corpus grande. Como referencia, pueden utilizarse 50, 100, 250, 500 y 1.000 documentos, ampliándolos si el equipo lo permite.

Para cada tamaño $n$:

1. Ejecute una misma consulta mediante búsqueda secuencial.
2. Repítala 30 veces y calcule el tiempo promedio.
3. Construya el índice invertido correspondiente a ese subconjunto.
4. Ejecute la consulta mediante el índice, también 30 veces.
5. Registre ambos tiempos y el número total de tokens.

Elabore una **gráfica del tiempo medio de ejecución en función del número de documentos**, colocando el número de documentos en el eje X y el tiempo en milisegundos en el eje Y. Represente los dos métodos en la misma gráfica.

**9. Estimación de una función de tiempo**

Utilizando los resultados experimentales, determine una función:

$t=f(n)$

para cada método de recuperación.

Como primera aproximación, utilice un ajuste lineal por mínimos cuadrados:

$t(n)=an+b$

Estime los parámetros $a$ y $b$, reporte el coeficiente de determinación $R^2$ y represente las funciones ajustadas junto a los datos experimentales.

Explique si el modelo lineal representa adecuadamente las mediciones y cómo se relaciona con la complejidad computacional de cada algoritmo.